# 🎫 Notebook 1: Seat Selection & Locking

When thousands of fans rush to buy concert tickets at the same time, how do we make sure two
people don't end up with the **same seat**? This is one of the hardest problems in ticket
booking systems — and it all comes down to **concurrency control**.

## 🎯 Learning Objectives

By the end of this notebook you will understand:

1. **How seat maps work** — venues store a JSON structure that describes every section, row, and seat. Each seat becomes a ticket row in the database.
2. **Why concurrent access is dangerous** — two users can read the same seat as "available" and both think they got it.
3. **PostgreSQL row-level locking (`SELECT ... FOR UPDATE`)** — a pessimistic strategy that makes the second user wait.
4. **Optimistic concurrency control (OCC)** — a lightweight strategy that lets everyone try, but only the first write wins.

Let's dive in! 🚀

## 🛠️ Setup

### 1. Start the infrastructure

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

This starts:
- **PostgreSQL** on port `5433` — our main database with events, venues, and tickets
- **Redis** on port `6380` — we'll use this in later notebooks for distributed locks
- **Adminer** on port `8081` — a web UI to browse the database
- **RedisInsight** on port `5541` — a web UI to browse Redis

### 2. Select the `.venv` kernel

In VS Code, click the kernel picker (top-right of this notebook) and select the `.venv` environment.
If it doesn't appear, reload the window (`Cmd+Shift+P` → "Reload Window").

### 3. Explore the data

- Open [Adminer → localhost:8081](http://localhost:8081) (server: `postgres`, user: `demo`, password: `demo`, db: `ticketmaster`)
- Open [RedisInsight → localhost:5541](http://localhost:5541) and add a connection to `redis:6379`

In [ ]:
# ============================================================
# Imports & Connection Helpers
# ============================================================

import psycopg2
import psycopg2.extras
import redis
import time
import json
import threading
from tabulate import tabulate

# --- Database configuration ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "dbname": "ticketmaster",
    "user": "demo",
    "password": "demo",
}

# --- Redis configuration ---
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6380,
    "decode_responses": True,
}


def get_db_connection():
    """Create a new PostgreSQL connection."""
    return psycopg2.connect(**DB_CONFIG)


def get_redis_client():
    """Create a new Redis client."""
    return redis.Redis(**REDIS_CONFIG)


# --- Test both connections ---
try:
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT version();")
    print(f"✅ PostgreSQL connected: {cur.fetchone()[0][:30]}...")
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print(f"✅ Redis connected: {r.info('server')['redis_version']}")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")

## 🏟️ Understanding the Seat Map

Every venue in our system has a **seat map** — a JSON structure that describes the physical
layout of the venue. Think of it like a blueprint:

```
Venue
 └── sections (FLOOR, LOWER, UPPER)
      └── rows (A, B, C, ...)
           └── seats (1, 2, 3, ...)
```

When an event is created, our system reads the venue's `seat_map` JSON and creates **one
ticket row per seat** in the `tickets` table. Each ticket starts with `status = 'available'`.

The client application (web/mobile) fetches this seat map and renders an **interactive seat
picker** — you've seen this on Ticketmaster, where you click on individual seats. Behind the
scenes, each seat maps to a ticket ID.

Let's look at the data for **Event 1: The Eras Tour - NYC** at **Madison Square Garden**.

In [ ]:
# ============================================================
# Explore the seat map for Event 1 — "The Eras Tour - NYC"
# ============================================================

conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Fetch the event details along with the venue seat map
cur.execute("""
    SELECT e.id AS event_id, e.name AS event_name, e.event_date,
           v.name AS venue_name, v.city, v.capacity, v.seat_map
    FROM events e
    JOIN venues v ON e.venue_id = v.id
    WHERE e.id = 1;
""")
event = cur.fetchone()

print(f"🎤 Event:  {event['event_name']}")
print(f"🏟️  Venue:  {event['venue_name']} ({event['city']})")
print(f"📅 Date:   {event['event_date']}")
print()

# Show the seat map structure
seat_map = event['seat_map']
print("📐 Seat Map Structure:")
print(json.dumps(seat_map, indent=2))
print()

# Count tickets per section
cur.execute("""
    SELECT section,
           COUNT(*) AS total_seats,
           COUNT(*) FILTER (WHERE status = 'available') AS available,
           COUNT(*) FILTER (WHERE status = 'sold') AS sold
    FROM tickets
    WHERE event_id = 1
    GROUP BY section
    ORDER BY section;
""")
rows = cur.fetchall()

print("🎟️ Ticket Availability by Section:")
print(tabulate(
    [[r['section'], r['total_seats'], r['available'], r['sold']] for r in rows],
    headers=["Section", "Total", "Available", "Sold"],
    tablefmt="simple_grid"
))

cur.close()
conn.close()

## 💥 The Double Booking Problem

Now here's the scary part. Imagine two fans — **Alice** and **Bob** — both click on the
**same seat** at almost the same time. Here's what happens with a naive implementation:

```
Timeline         Alice (User 101)                 Bob (User 102)
─────────────────────────────────────────────────────────────────
  T1              SELECT status FROM tickets       
                  WHERE id = 42;                   
                  → status = 'available' ✅        
                                                   
  T2                                               SELECT status FROM tickets
                                                   WHERE id = 42;
                                                   → status = 'available' ✅
                                                   
  T3              UPDATE tickets                   
                  SET status = 'sold'              
                  WHERE id = 42;                   
                  → Alice thinks she got it ✅     
                                                   
  T4                                               UPDATE tickets
                                                   SET status = 'sold'
                                                   WHERE id = 42;
                                                   → Bob ALSO thinks he got it ✅
─────────────────────────────────────────────────────────────────
                  ⚠️ DOUBLE BOOKING! Both users were told they got the seat.
```

The problem is the **gap between reading and writing**. Both users see the seat as available
because they read BEFORE the other user writes. This is called a **race condition** — the
outcome depends on who finishes first.

Let's reproduce this bug in real code. 👇

In [ ]:
# ============================================================
# 💥 Demonstrate the race condition (UNSAFE booking)
# ============================================================

# Pick an available ticket to use for this demo
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' LIMIT 1;")
demo_ticket_id = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Using ticket ID {demo_ticket_id} for the race condition demo\n")

# Store results from each thread
results = {}


def book_ticket_unsafe(user_id, ticket_id):
    """
    UNSAFE booking function — DO NOT use in production!
    
    This reads the ticket status first, then updates it.
    The gap between READ and WRITE is where the race condition lives.
    """
    conn = get_db_connection()
    conn.autocommit = True  # Each statement commits immediately (no transaction protection)
    cur = conn.cursor()
    try:
        # Step 1: Read the current status
        cur.execute("SELECT status FROM tickets WHERE id = %s;", (ticket_id,))
        status = cur.fetchone()[0]

        # Simulate network delay / slow processing
        time.sleep(0.1)

        if status == 'available':
            # Step 2: Update the ticket — but someone else may have updated it already!
            cur.execute("UPDATE tickets SET status = 'sold' WHERE id = %s;", (ticket_id,))
            results[user_id] = "✅ BOOKED (thinks they got it)"
        else:
            results[user_id] = "❌ REJECTED (seat taken)"
    finally:
        cur.close()
        conn.close()


# Launch two threads trying to book the SAME ticket at the SAME time
t1 = threading.Thread(target=book_ticket_unsafe, args=(101, demo_ticket_id))
t2 = threading.Thread(target=book_ticket_unsafe, args=(102, demo_ticket_id))

t1.start()
t2.start()
t1.join()
t2.join()

# Show results
print("Results:")
for user_id, result in sorted(results.items()):
    print(f"  User {user_id}: {result}")

print()
print("💥 DOUBLE BOOKING! Both users think they booked the same seat.")
print("   In a real system, one user would show up and find someone else in their seat.")

# Reset the ticket for the next demo
conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s;", (demo_ticket_id,))
conn.commit()
cur.close()
conn.close()
print(f"\n🔄 Reset ticket {demo_ticket_id} back to 'available' for next demo.")

## 🔒 Fix #1: Pessimistic Locking (`SELECT ... FOR UPDATE`)

The first fix is called **pessimistic locking** — we *assume* conflicts will happen, so we
lock the row **before** reading it.

PostgreSQL's `SELECT ... FOR UPDATE` does exactly this:
- It reads the row **and** takes an exclusive lock on it.
- Any other transaction that tries to `SELECT ... FOR UPDATE` the **same row** will **block**
  (wait) until the first transaction commits or rolls back.

### How it works

```
Timeline         Alice (User 101)                 Bob (User 102)
─────────────────────────────────────────────────────────────────
  T1              BEGIN;
                  SELECT ... FOR UPDATE
                  WHERE id = 42;
                  → status = 'available' 🔒

  T2                                               BEGIN;
                                                   SELECT ... FOR UPDATE
                                                   WHERE id = 42;
                                                   → ⏳ BLOCKED (waiting for Alice's lock)

  T3              UPDATE status = 'sold';
                  COMMIT;
                  → Alice got the seat! ✅

  T4                                               → Lock released! Bob can read now.
                                                   → status = 'sold' ❌
                                                   ROLLBACK;
─────────────────────────────────────────────────────────────────
                  ✅ Only ONE user gets the seat. No double booking!
```

The key insight: `FOR UPDATE` turns the gap between read and write into an **atomic
operation** by holding a lock across the entire transaction.

In [ ]:
# ============================================================
# 🔒 Pessimistic Locking — SELECT ... FOR UPDATE
# ============================================================

# Pick a fresh available ticket
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' LIMIT 1;")
demo_ticket_id = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Using ticket ID {demo_ticket_id} for the pessimistic locking demo\n")

results = {}


def book_ticket_pessimistic(user_id, ticket_id):
    """
    SAFE booking using pessimistic locking.
    
    SELECT ... FOR UPDATE locks the row so only one transaction
    can read+write the ticket at a time.
    """
    conn = get_db_connection()
    conn.autocommit = False  # We need explicit transaction control
    cur = conn.cursor()
    try:
        # Step 1: Read AND lock the row in one statement
        cur.execute(
            "SELECT status FROM tickets WHERE id = %s FOR UPDATE;",
            (ticket_id,)
        )
        status = cur.fetchone()[0]

        # Simulate processing time
        time.sleep(0.1)

        if status == 'available':
            cur.execute(
                "UPDATE tickets SET status = 'sold' WHERE id = %s;",
                (ticket_id,)
            )
            conn.commit()
            results[user_id] = "✅ BOOKED"
        else:
            conn.rollback()
            results[user_id] = "❌ REJECTED (seat already taken)"
    except Exception as e:
        conn.rollback()
        results[user_id] = f"❌ ERROR: {e}"
    finally:
        cur.close()
        conn.close()


# Launch two threads — same ticket, same time
t1 = threading.Thread(target=book_ticket_pessimistic, args=(101, demo_ticket_id))
t2 = threading.Thread(target=book_ticket_pessimistic, args=(102, demo_ticket_id))

t1.start()
t2.start()
t1.join()
t2.join()

# Show results
print("Results:")
for user_id, result in sorted(results.items()):
    print(f"  User {user_id}: {result}")

# Verify in the database
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT status FROM tickets WHERE id = %s;", (demo_ticket_id,))
final_status = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n🔍 Final ticket status in DB: '{final_status}'")
print("✅ Exactly ONE user got the seat. The other was safely rejected.")

# Reset for next demo
conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s;", (demo_ticket_id,))
conn.commit()
cur.close()
conn.close()
print(f"🔄 Reset ticket {demo_ticket_id} back to 'available'.")

## ⚡ Fix #2: Optimistic Concurrency Control (OCC)

Pessimistic locking works, but it has a cost: the second user **blocks** (waits) until the
first transaction finishes. Under heavy load (thousands of users hitting the same hot seats),
this can cause timeouts and slow responses.

**Optimistic Concurrency Control** takes a different approach:
- Don't lock anything upfront.
- Just try to update the row with a **condition** — `WHERE status = 'available'`.
- If the update affects 0 rows, someone else got there first. No harm done.

### How it works

```sql
-- Instead of SELECT then UPDATE, do it in ONE atomic UPDATE:
UPDATE tickets
SET status = 'sold'
WHERE id = 42 AND status = 'available'
RETURNING id;
```

- If a row is returned → you got the seat! 🎉
- If no row is returned → someone else already changed the status. Try again or show an error.

### Why is this "optimistic"?

Because we **optimistically assume** there won't be a conflict. We don't lock anything — we
just check at write time. If we're wrong, we handle it gracefully.

This is faster because **losing threads don't block** — they fail immediately and can retry
or show an error.

In [ ]:
# ============================================================
# ⚡ Optimistic Concurrency Control — UPDATE ... WHERE status = 'available'
# ============================================================

# Pick a fresh available ticket
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' LIMIT 1;")
demo_ticket_id = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Using ticket ID {demo_ticket_id} for the optimistic locking demo\n")

results = {}


def book_ticket_optimistic(user_id, ticket_id):
    """
    SAFE booking using optimistic concurrency.
    
    A single UPDATE with a WHERE condition ensures only one user
    can flip the status from 'available' to 'sold'.
    """
    conn = get_db_connection()
    conn.autocommit = False
    cur = conn.cursor()
    try:
        # Simulate reading the seat map / processing time
        time.sleep(0.1)

        # One atomic UPDATE — the WHERE clause is the "optimistic check"
        cur.execute(
            "UPDATE tickets SET status = 'sold' WHERE id = %s AND status = 'available' RETURNING id;",
            (ticket_id,)
        )
        row = cur.fetchone()

        if row:
            conn.commit()
            results[user_id] = "✅ BOOKED"
        else:
            conn.rollback()
            results[user_id] = "❌ REJECTED (seat already taken)"
    except Exception as e:
        conn.rollback()
        results[user_id] = f"❌ ERROR: {e}"
    finally:
        cur.close()
        conn.close()


# Launch two threads — same ticket, same time
t1 = threading.Thread(target=book_ticket_optimistic, args=(101, demo_ticket_id))
t2 = threading.Thread(target=book_ticket_optimistic, args=(102, demo_ticket_id))

t1.start()
t2.start()
t1.join()
t2.join()

# Show results
print("Results:")
for user_id, result in sorted(results.items()):
    print(f"  User {user_id}: {result}")

# Verify in the database
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT status FROM tickets WHERE id = %s;", (demo_ticket_id,))
final_status = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n🔍 Final ticket status in DB: '{final_status}'")
print("✅ Exactly ONE user got the seat. The loser failed instantly — no waiting!")

# Reset for next demo
conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s;", (demo_ticket_id,))
conn.commit()
cur.close()
conn.close()
print(f"🔄 Reset ticket {demo_ticket_id} back to 'available'.")

## 📊 Comparing the Approaches

Let's run a real benchmark: **10 threads** all trying to book the **same ticket** at once.

- With **pessimistic locking**, 9 threads will queue up and wait for the lock.
- With **optimistic concurrency**, 9 threads will fail immediately (no waiting).

We'll measure the total time for all threads to finish.

In [ ]:
# ============================================================
# 📊 Benchmark: Pessimistic vs Optimistic with 10 threads
# ============================================================

NUM_THREADS = 10


def run_benchmark(booking_fn, label):
    """
    Run a booking function with NUM_THREADS threads all targeting the same ticket.
    Returns (elapsed_time, results_dict).
    """
    # Get a fresh available ticket
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' LIMIT 1;")
    ticket_id = cur.fetchone()[0]
    cur.close()
    conn.close()

    thread_results = {}

    def wrapper(user_id):
        booking_fn(user_id, ticket_id, thread_results)

    threads = [threading.Thread(target=wrapper, args=(i,)) for i in range(NUM_THREADS)]

    start = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    elapsed = time.time() - start

    # Count outcomes
    booked = sum(1 for v in thread_results.values() if "BOOKED" in v)
    rejected = sum(1 for v in thread_results.values() if "REJECTED" in v)

    # Reset the ticket
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s;", (ticket_id,))
    conn.commit()
    cur.close()
    conn.close()

    return elapsed, booked, rejected


def pessimistic_for_bench(user_id, ticket_id, results_dict):
    conn = get_db_connection()
    conn.autocommit = False
    cur = conn.cursor()
    try:
        cur.execute("SELECT status FROM tickets WHERE id = %s FOR UPDATE;", (ticket_id,))
        status = cur.fetchone()[0]
        time.sleep(0.05)  # Simulate processing
        if status == 'available':
            cur.execute("UPDATE tickets SET status = 'sold' WHERE id = %s;", (ticket_id,))
            conn.commit()
            results_dict[user_id] = "✅ BOOKED"
        else:
            conn.rollback()
            results_dict[user_id] = "❌ REJECTED"
    except Exception as e:
        conn.rollback()
        results_dict[user_id] = f"❌ ERROR: {e}"
    finally:
        cur.close()
        conn.close()


def optimistic_for_bench(user_id, ticket_id, results_dict):
    conn = get_db_connection()
    conn.autocommit = False
    cur = conn.cursor()
    try:
        time.sleep(0.05)  # Simulate processing
        cur.execute(
            "UPDATE tickets SET status = 'sold' WHERE id = %s AND status = 'available' RETURNING id;",
            (ticket_id,)
        )
        row = cur.fetchone()
        if row:
            conn.commit()
            results_dict[user_id] = "✅ BOOKED"
        else:
            conn.rollback()
            results_dict[user_id] = "❌ REJECTED"
    except Exception as e:
        conn.rollback()
        results_dict[user_id] = f"❌ ERROR: {e}"
    finally:
        cur.close()
        conn.close()


# Run benchmarks
print(f"🏁 Racing {NUM_THREADS} threads for the same ticket...\n")

pess_time, pess_booked, pess_rejected = run_benchmark(pessimistic_for_bench, "Pessimistic")
opt_time, opt_booked, opt_rejected = run_benchmark(optimistic_for_bench, "Optimistic")

print(tabulate(
    [
        ["Pessimistic (FOR UPDATE)", f"{pess_time:.3f}s", pess_booked, pess_rejected],
        ["Optimistic (WHERE check)", f"{opt_time:.3f}s", opt_booked, opt_rejected],
    ],
    headers=["Strategy", "Total Time", "Booked", "Rejected"],
    tablefmt="simple_grid"
))

print(f"\n⚡ Optimistic was ~{pess_time / opt_time:.1f}x faster!")
print("   Pessimistic is slower because losing threads wait in line for the lock.")
print("   Optimistic lets losing threads fail immediately — no blocking.")

## 🎟️ Booking Multiple Seats

In practice, users don't just book one seat — they book **2, 4, or 6 seats together** for
friends and family. This makes things more complicated:

- All selected seats must be booked **atomically** — either ALL succeed or NONE do.
- If we can't get all the seats, we must release any we've already locked.

### The Deadlock Danger ⚠️

When locking multiple rows, there's a risk of **deadlock**:

```
Thread A: locks ticket 1, then tries to lock ticket 2
Thread B: locks ticket 2, then tries to lock ticket 1
→ Both are waiting for each other forever! 💀
```

The fix is simple: **always lock rows in a consistent order** (e.g., sorted by ticket ID).
This way, all threads acquire locks in the same order and deadlocks can't happen.

```sql
-- Lock multiple seats in ID order to prevent deadlocks
SELECT id, status FROM tickets
WHERE id IN (42, 43, 44)
ORDER BY id
FOR UPDATE;
```

In [ ]:
# ============================================================
# 🎟️ Booking multiple seats atomically
# ============================================================


def book_multiple_seats(user_id, ticket_ids):
    """
    Book multiple seats in a single atomic transaction.
    
    Uses SELECT ... FOR UPDATE with ORDER BY id to prevent deadlocks.
    If any seat is unavailable, the entire booking is rolled back.
    """
    conn = get_db_connection()
    conn.autocommit = False
    cur = conn.cursor()
    try:
        # Sort IDs to prevent deadlocks — every thread locks in the same order
        sorted_ids = sorted(ticket_ids)

        # Lock all requested tickets at once, ordered by ID
        cur.execute(
            "SELECT id, section, row_label, seat_number, price, status "
            "FROM tickets WHERE id = ANY(%s) ORDER BY id FOR UPDATE;",
            (sorted_ids,)
        )
        tickets = cur.fetchall()

        # Check that we got all tickets and all are available
        if len(tickets) != len(sorted_ids):
            conn.rollback()
            return False, "Some ticket IDs don't exist"

        unavailable = [t for t in tickets if t[5] != 'available']
        if unavailable:
            conn.rollback()
            seat_names = [f"{t[1]}-{t[2]}{t[3]}" for t in unavailable]
            return False, f"Seats already taken: {', '.join(seat_names)}"

        # All seats are available — mark them as sold
        cur.execute(
            "UPDATE tickets SET status = 'sold' WHERE id = ANY(%s);",
            (sorted_ids,)
        )

        # Calculate total price
        total_price = sum(t[4] for t in tickets)

        # Create a booking record
        cur.execute(
            "INSERT INTO bookings (user_id, event_id, total_price, status) "
            "VALUES (%s, %s, %s, 'confirmed') RETURNING id;",
            (user_id, tickets[0][0], total_price)  # event_id from first ticket
        )
        booking_id = cur.fetchone()[0]

        # Link tickets to the booking
        for t in tickets:
            cur.execute(
                "INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s);",
                (booking_id, t[0])
            )

        conn.commit()

        seat_names = [f"{t[1]}-{t[2]}{t[3]}" for t in tickets]
        return True, f"Booking #{booking_id}: {', '.join(seat_names)} (${total_price})"

    except Exception as e:
        conn.rollback()
        return False, f"Error: {e}"
    finally:
        cur.close()
        conn.close()


# Find 3 available tickets in the same section
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT id, section, row_label, seat_number
    FROM tickets
    WHERE event_id = 1 AND status = 'available' AND section = 'FLOOR'
    ORDER BY row_label, seat_number
    LIMIT 3;
""")
available_tickets = cur.fetchall()
cur.close()
conn.close()

if len(available_tickets) < 3:
    print("⚠️ Not enough available FLOOR tickets. Try resetting the database.")
else:
    ticket_ids = [t[0] for t in available_tickets]
    seat_labels = [f"{t[1]}-{t[2]}{t[3]}" for t in available_tickets]
    print(f"🎯 Trying to book 3 seats: {', '.join(seat_labels)}")
    print(f"   Ticket IDs: {ticket_ids}\n")

    success, message = book_multiple_seats(user_id=201, ticket_ids=ticket_ids)

    if success:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")

    # Show updated ticket status
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        "SELECT id, section, row_label, seat_number, status FROM tickets WHERE id = ANY(%s);",
        (ticket_ids,)
    )
    rows = cur.fetchall()
    print("\n🔍 Ticket status after booking:")
    print(tabulate(
        rows,
        headers=["ID", "Section", "Row", "Seat", "Status"],
        tablefmt="simple_grid"
    ))
    cur.close()
    conn.close()

In [ ]:
# ============================================================
# 🧹 Cleanup — Reset demo data
# ============================================================
# Reset any tickets we changed during this notebook back to 'available'
# and delete any test bookings we created.

conn = get_db_connection()
cur = conn.cursor()

# Delete test booking_tickets and bookings (user_ids we used: 101, 102, 201)
cur.execute("""
    DELETE FROM booking_tickets
    WHERE booking_id IN (
        SELECT id FROM bookings WHERE user_id IN (101, 102, 201)
    );
""")
deleted_bt = cur.rowcount

cur.execute("DELETE FROM bookings WHERE user_id IN (101, 102, 201);")
deleted_bookings = cur.rowcount

# Reset any 'sold' tickets that were originally available (from our demos)
# We'll reset all event_id=1 tickets that were sold by our test users
# back to 'available'. The originally-sold tickets from init.sql are fine
# because we only touched 'available' ones.
cur.execute("""
    UPDATE tickets SET status = 'available'
    WHERE event_id = 1 AND status = 'sold'
    AND id NOT IN (
        SELECT ticket_id FROM booking_tickets
    );
""")
reset_tickets = cur.rowcount

conn.commit()
cur.close()
conn.close()

print("🧹 Cleanup complete!")
print(f"   Deleted {deleted_bt} booking_ticket links")
print(f"   Deleted {deleted_bookings} test bookings")
print(f"   Reset {reset_tickets} tickets back to 'available'")

## 📝 Summary

We explored three approaches to handling concurrent seat bookings:

| Approach | How it works | Pros | Cons |
|----------|-------------|------|------|
| **Naive (no protection)** | Read status, then update | Simple | ❌ Double bookings! |
| **Pessimistic (`FOR UPDATE`)** | Lock the row before reading | ✅ Guaranteed safe | Losers wait (slow under load) |
| **Optimistic (`WHERE` check)** | Conditional update, check rows affected | ✅ Safe + fast for losers | May need retry logic |

### When to use which?

- **Pessimistic locking** is best when conflicts are **frequent** and you need to do complex
  work between read and write (e.g., booking multiple seats, checking business rules).
- **Optimistic concurrency** is best when conflicts are **rare** or when you want the fastest
  possible failure for losers (e.g., flash sales with many competing buyers).
- In a real Ticketmaster-like system, you'd likely use **both**: pessimistic locking for the
  multi-seat booking flow, and optimistic checks for individual seat grabs.

### 🔮 What's Next?

In **Notebook 2**, we'll tackle **flash sales** — what happens when 100,000 users all hit
"Buy" at the exact same second? We'll explore:
- Temporary seat holds with Redis TTLs
- Queue-based booking to smooth out traffic spikes
- Distributed locks for horizontal scaling